# Tutorial 23: LangChain 1.0 Utilities

In this tutorial we cover the quality-of-life utilities introduced in LangChain v0.3 and 1.0: `init_chat_model()` for provider-agnostic model initialisation, `trim_messages()` / `filter_messages()` for context window management, and the move to dedicated integration packages.

## 1. Why These Utilities?

Before v0.3, switching model providers required changing import statements and constructor calls throughout your code. Managing long conversation histories was manual. LangChain 1.0 addresses both:

- `init_chat_model()` — one universal constructor for any supported provider
- `trim_messages()` — declarative context window budgeting
- `filter_messages()` — select messages by role, type, or id
- `merge_message_runs()` — collapse consecutive same-role messages

## 2. Setup

In [1]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage,
    trim_messages, filter_messages, merge_message_runs
)
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. `init_chat_model()` — Provider-Agnostic Initialisation

The format is `provider:model_name`. This function returns a standard `BaseChatModel` compatible with all LangChain components — LCEL chains, LangGraph nodes, tools, and more.

In [2]:
# All of these return the same BaseChatModel interface
# Uncomment the provider you want to use

# Groq (used in this tutorial series)
llm = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.1)

# OpenAI (requires OPENAI_API_KEY)
# llm = init_chat_model("openai:gpt-4o", temperature=0.1)

# Anthropic (requires ANTHROPIC_API_KEY)
# llm = init_chat_model("anthropic:claude-sonnet-4-6", temperature=0.1)

# Ollama (local, no API key needed)
# llm = init_chat_model("ollama:llama3", temperature=0.1)

print(f"Model type: {type(llm).__name__}")
print("Same interface regardless of provider — all downstream code works unchanged.")

Model type: ChatGroq
Same interface regardless of provider — all downstream code works unchanged.


In [3]:
# Works in LCEL chains
from langchain_core.output_parsers import StrOutputParser

chain = ChatPromptTemplate.from_template("Tell me a one-sentence fact about {topic}.") | llm | StrOutputParser()
print(chain.invoke({"topic": "the Moon"}))

The Moon is in a synchronous rotation with Earth, meaning it takes the Moon the same amount of time to rotate on its axis as it takes to orbit Earth, resulting in the same side of the Moon always facing our planet.


In [4]:
# Configurable model — override at invocation time without changing the chain
configurable_llm = init_chat_model(temperature=0.1)  # no model specified = fully configurable

# Specify at runtime
resp = configurable_llm.invoke(
    [HumanMessage(content="What is 2+2?")],
    config={"configurable": {"model": "groq:llama-3.1-8b-instant"}}
)
print(f"Configurable model response: {resp.content}")

Configurable model response: 2 + 2 = 4.


## 4. Dedicated Integration Packages

In LangChain v0.3+, each provider has its own package instead of living in `langchain-community`.

In [5]:
# Old style (langchain-community, still works but not recommended)
# from langchain.chat_models import ChatOpenAI

# New style — dedicated packages
migration_guide = {
    "langchain-openai": "from langchain_openai import ChatOpenAI",
    "langchain-anthropic": "from langchain_anthropic import ChatAnthropic",
    "langchain-groq": "from langchain_groq import ChatGroq",
    "langchain-google-genai": "from langchain_google_genai import ChatGoogleGenerativeAI",
    "langchain-ollama": "from langchain_ollama import ChatOllama",
    "langchain-aws": "from langchain_aws import ChatBedrock",
}

print("Provider → Dedicated Package Migration:")
for pkg, import_stmt in migration_guide.items():
    print(f"  pip install {pkg:30s}  →  {import_stmt}")

Provider → Dedicated Package Migration:
  pip install langchain-openai                →  from langchain_openai import ChatOpenAI
  pip install langchain-anthropic             →  from langchain_anthropic import ChatAnthropic
  pip install langchain-groq                  →  from langchain_groq import ChatGroq
  pip install langchain-google-genai          →  from langchain_google_genai import ChatGoogleGenerativeAI
  pip install langchain-ollama                →  from langchain_ollama import ChatOllama
  pip install langchain-aws                   →  from langchain_aws import ChatBedrock


## 5. `trim_messages()` — Context Window Management

As conversations grow, message history can exceed the model's context window. `trim_messages()` keeps the most recent messages within a token budget.

In [6]:
from langchain_core.messages import trim_messages

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Tell me about Python.", id="h1"),
    AIMessage(content="Python is a versatile programming language widely used in data science and web development.", id="a1"),
    HumanMessage(content="What about LangChain?", id="h2"),
    AIMessage(content="LangChain is a framework for building applications powered by language models.", id="a2"),
    HumanMessage(content="And LangGraph?", id="h3"),
    AIMessage(content="LangGraph extends LangChain for stateful, graph-based multi-agent workflows.", id="a3"),
    HumanMessage(content="What is 2+2?", id="h4"),
]

# Keep last 3 messages within a 100-token budget
trimmed_last = trim_messages(
    messages,
    max_tokens=150,
    strategy="last",          # keep the most recent messages
    token_counter=llm,        # use the model's tokeniser
    include_system=True,      # always keep the system message
    allow_partial=False,      # never cut a message in half
    start_on="human",         # ensure the first message is from the human
)

print(f"Original: {len(messages)} messages")
print(f"Trimmed:  {len(trimmed_last)} messages")
print("\nKept messages:")
for msg in trimmed_last:
    role = msg.__class__.__name__
    print(f"  {role}: {msg.content[:80]}")

/home/domenico/clone/langchain-langgraph-tutorial/.venv/lib/python3.12/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


Original: 8 messages
Trimmed:  8 messages

Kept messages:
  SystemMessage: You are a helpful assistant.
  HumanMessage: Tell me about Python.
  AIMessage: Python is a versatile programming language widely used in data science and web d
  HumanMessage: What about LangChain?
  AIMessage: LangChain is a framework for building applications powered by language models.
  HumanMessage: And LangGraph?
  AIMessage: LangGraph extends LangChain for stateful, graph-based multi-agent workflows.
  HumanMessage: What is 2+2?


In [7]:
# Strategy "first" — keep oldest messages (useful for preserving context from start of session)
trimmed_first = trim_messages(
    messages,
    max_tokens=150,
    strategy="first",
    token_counter=llm,
)

print(f"Strategy 'first' keeps {len(trimmed_first)} messages:")
for msg in trimmed_first:
    print(f"  {msg.__class__.__name__}: {msg.content[:60]}")

Strategy 'first' keeps 8 messages:
  SystemMessage: You are a helpful assistant.
  HumanMessage: Tell me about Python.
  AIMessage: Python is a versatile programming language widely used in da
  HumanMessage: What about LangChain?
  AIMessage: LangChain is a framework for building applications powered b
  HumanMessage: And LangGraph?
  AIMessage: LangGraph extends LangChain for stateful, graph-based multi-
  HumanMessage: What is 2+2?


## 6. `filter_messages()` — Select by Role or ID

In [8]:
from langchain_core.messages import filter_messages

# Keep only human messages
human_only = filter_messages(messages, include_types=["human"])
print(f"Human messages only ({len(human_only)}):")
for msg in human_only:
    print(f"  - {msg.content}")

print()

# Exclude system messages
no_system = filter_messages(messages, exclude_types=["system"])
print(f"Excluding system ({len(no_system)} messages):")
for msg in no_system:
    print(f"  {msg.__class__.__name__}: {msg.content[:60]}")

print()

# Keep specific messages by ID
selected = filter_messages(messages, include_ids=["h1", "a1", "h4"])
print(f"Selected by ID ({len(selected)} messages):")
for msg in selected:
    print(f"  [{msg.id}] {msg.__class__.__name__}: {msg.content[:60]}")

Human messages only (4):
  - Tell me about Python.
  - What about LangChain?
  - And LangGraph?
  - What is 2+2?

Excluding system (7 messages):
  HumanMessage: Tell me about Python.
  AIMessage: Python is a versatile programming language widely used in da
  HumanMessage: What about LangChain?
  AIMessage: LangChain is a framework for building applications powered b
  HumanMessage: And LangGraph?
  AIMessage: LangGraph extends LangChain for stateful, graph-based multi-
  HumanMessage: What is 2+2?

Selected by ID (3 messages):
  [h1] HumanMessage: Tell me about Python.
  [a1] AIMessage: Python is a versatile programming language widely used in da
  [h4] HumanMessage: What is 2+2?


## 7. `merge_message_runs()` — Collapse Consecutive Same-Role Messages

In [9]:
from langchain_core.messages import merge_message_runs

# Happens when tool results arrive as separate messages or prompts are assembled from parts
fragmented = [
    SystemMessage(content="You are helpful."),
    SystemMessage(content="Always be concise."),
    HumanMessage(content="First question: what is Python?"),
    HumanMessage(content="Second question: what is LangChain?"),
    AIMessage(content="Python is a programming language."),
    AIMessage(content="LangChain is an LLM framework."),
]

merged = merge_message_runs(fragmented)

print(f"Before merge: {len(fragmented)} messages")
print(f"After merge:  {len(merged)} messages\n")
for msg in merged:
    print(f"{msg.__class__.__name__}:")
    print(f"  {msg.content}")

Before merge: 6 messages
After merge:  3 messages

SystemMessage:
  You are helpful.
Always be concise.
HumanMessage:
  First question: what is Python?
Second question: what is LangChain?
AIMessage:
  Python is a programming language.
LangChain is an LLM framework.


## 8. Structured Output with `.with_structured_output()`

In [10]:
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: int = Field(description="Rating from 1 to 10", ge=1, le=10)
    summary: str = Field(description="One-sentence summary of the review")
    recommended: bool = Field(description="Whether you recommend this movie")


# .with_structured_output() works the same across all providers
structured_llm = llm.with_structured_output(MovieReview)

review = structured_llm.invoke("Review the movie Inception (2010) in one sentence.")
print(f"Title: {review.title}")
print(f"Rating: {review.rating}/10")
print(f"Summary: {review.summary}")
print(f"Recommended: {review.recommended}")

Title: Inception
Rating: 8/10
Summary: A mind-bending sci-fi action film that delves into the concept of shared dreaming.
Recommended: True


## 9. Putting It Together — Context-Aware Chatbot

A chatbot that uses `trim_messages()` to stay within its context window and `init_chat_model()` to remain provider-agnostic.

In [11]:
from typing import List, Annotated, TypedDict
from langgraph.graph import StateGraph, END
import operator

from langchain_core.messages import BaseMessage

chat_model = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.1)


class ContextAwareState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]


def context_aware_chat(state: ContextAwareState) -> ContextAwareState:
    # Trim messages to fit within 500 tokens before sending to LLM
    trimmed = trim_messages(
        state["messages"],
        max_tokens=500,
        strategy="last",
        token_counter=chat_model,
        include_system=True,
        allow_partial=False,
        start_on="human",
    )
    response = chat_model.invoke(trimmed)
    return {"messages": [AIMessage(content=response.content)]}


from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage

ca_workflow = StateGraph(ContextAwareState)
ca_workflow.add_node("chat", context_aware_chat)
ca_workflow.set_entry_point("chat")
ca_workflow.add_edge("chat", END)
ca_app = ca_workflow.compile(checkpointer=MemorySaver())

config = {"configurable": {"thread_id": "context-aware-1"}}

system = SystemMessage(content="You are a concise assistant.")

for user_msg in ["What is Python?", "And LangChain?", "And LangGraph?"]:
    r = ca_app.invoke({"messages": [system, HumanMessage(content=user_msg)]}, config)
    print(f"Q: {user_msg}")
    print(f"A: {r['messages'][-1].content[:120]}\n")

Q: What is Python?
A: **Python Overview**

Python is a high-level, interpreted programming language that is widely used for various purposes, 

Q: And LangChain?
A: **LangChain Overview**

LangChain is a Python library for building conversational AI applications. It provides a simple 

Q: And LangGraph?
A: I couldn't find any information on "LangGraph". It's possible that it's a lesser-known or newly created term. Can you pr



## 10. Conclusion

In this tutorial we covered LangChain 1.0 utilities:
- `init_chat_model("provider:model")` — universal, provider-agnostic model constructor
- Dedicated integration packages replacing `langchain-community`
- `trim_messages()` — declarative context window management with `strategy`, `max_tokens`, `token_counter`
- `filter_messages()` — select messages by type, role, or id
- `merge_message_runs()` — collapse consecutive same-role messages
- `.with_structured_output(schema)` — consistent structured output across providers

In Tutorial 24 we explore Agent Middleware — injecting logic at any point in the agent loop using `before_model`, `after_model`, and `modify_model_request`.